# Evaluating AGI: A Scaffold for Benchpress

I built this scaffold to test new AGI benchmarks using the `benchpress` matrix completion method. 

## The Problem
When proposing a new benchmark (especially in competitions), it's hard to know if it's actually measuring something **novel**, or just redundant with existing tests (like AIME or MMLU). [Benchpress](https://open.substack.com/pub/dimitrisp/p/you-dont-need-to-run-every-eval?utm_campaign=post-expanded-share&utm_medium=web) solves this by predicting how models *should* score based on the entire ecosystem of existing benchmarks. If your new benchmark's scores can be perfectly predicted, it's redundant. If they can't, you've found a novel axis of evaluation (or noise).

Here's how to use my setup to test your own benchmarks.

## 1. Installation

Instead of copying all the scaffold code, you can install the library directly from GitHub. This keeps your notebook clean and focuses strictly on the evaluation.

In [ ]:
!pip install -q git+https://github.com/kafkasl/evaluating-agi.git

## 2. Using the Scaffold

The core of the library is the `check_novelty` function. It takes a dictionary of scores your benchmark assigned to various models, and calculates a **Leave-One-Out (LOO) median error**.

*   `~3 pts median error`: **Redundant** (Highly predictable from other benchmarks)
*   `~8 pts median error`: **Somewhat Novel**
*   `~12+ pts median error`: **Novel** (Measures something genuinely new, e.g. ARC-AGI)
*   `~18+ pts median error`: **Noise** (Random or broken benchmark)

Let's import it and test a dummy benchmark.

In [ ]:
from evaluating_agi.benchpress import check_novelty

# Example: We ran our new benchmark "LogicPuzzle-100" on 4 major models.
# The keys must match the model IDs in the BenchPress matrix.
my_benchmark_scores = {
    'gpt-4.5': 85.0,
    'claude-3.7-sonnet': 82.5,
    'o3-mini-high': 88.0,
    'claude-opus-4': 78.0
}

# Evaluate how novel these scores are against the existing AGI landscape
report = check_novelty(my_benchmark_scores, name='LogicPuzzle-100')

## 3. Interpreting the Results

When you run the code above, the output will look something like this:

```text
  Model                        Actual   Pred   Err
  claude-opus-4                  78.0   74.2   3.8
  claude-3.7-sonnet              82.5   79.1   3.4
  gpt-4.5                        85.0   86.2   1.2
  o3-mini-high                   88.0   87.5   0.5
  Median LOO: 2.3 pts → redundant
```

In this example, the hypothetical benchmark scores were easily predicted based on how these models perform on existing benchmarks. This suggests that **LogicPuzzle-100 is redundant**—it correlates strongly with something we already measure (like MMLU or AIME), and doesn't tell us anything new about AGI capabilities.

### Why does this matter for the competition?
Because the competition lacks a definitive ranking mechanism for *new* benchmarks, we need a mathematical way to check if a benchmark is genuinely pushing the frontier. If your benchmark returns a `Median LOO: 14 pts → NOVEL`, you have solid proof that you're measuring an orthogonal skill.

## 4. Deep Dive

This notebook uses the high-level API for quick testing. If you want to see exactly how I built the scaffold, check out the source code and development notebooks on GitHub:

*   **[GitHub Repository](https://github.com/kafkasl/evaluating-agi)**
*   **[benchpress.ipynb](https://github.com/kafkasl/evaluating-agi/blob/main/nbs/benchpress.ipynb)** - The full matrix completion logic and methodology.

Happy benchmarking!